In [0]:
incremental_orders_df = (
    spark.read
         .option("header","true")
         .csv(
             "abfss://retailsales@harshadls01.dfs.core.windows.net/bronze/orders_incremental.csv"
         )
)

In [0]:
display(incremental_orders_df)

order_id,customer_id,product_id,quantity
1006,2,101,1
1007,4,104,3


## Cast Quantity

In [0]:
from pyspark.sql.functions import col

incremental_orders_df = (
    incremental_orders_df
    .withColumn(
        "quantity",
        col("quantity").cast("integer")
    )
)


## Load Existing Delta Table

In [0]:
from delta.tables import DeltaTable

orders_delta = DeltaTable.forPath(
    spark,
    "abfss://retailsales@harshadls01.dfs.core.windows.net/silver/orders_delta"
)

## Perform MERGE

In [0]:
orders_delta.alias("target").merge(
    incremental_orders_df.alias("source"),
    "target.order_id = source.order_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Verify Result

In [0]:
updated_orders = spark.read.format("delta").load(
    "abfss://retailsales@harshadls01.dfs.core.windows.net/silver/orders_delta"
)

display(updated_orders)

order_id,customer_id,product_id,quantity
1001,1,101,1
1004,1,104,2
1002,2,102,2
1003,3,103,3
1005,5,102,1
1006,2,101,1
1007,4,104,3
